# private-doc-qa: Colab pipeline v2 (customer profiles)

Same pipeline as notebook 01, plus what was added for white-label use: **customer profiles**, a **stricter prompt**, a **grounding check** (every figure in an answer must appear in the sources it cites) and a **supporting quote** next to each citation.

**Before you start:** Runtime > Change runtime type > **T4 GPU**. Use a personal Google account. Never upload real client or employer data here.

Colab sessions are temporary. The code lives in your GitHub repo; this notebook only clones it and calls it. Run the cells in order.

**What to look for:** the first version of this pipeline scored 31/34 on answerable questions. All 3 failures were "how many days notice" questions; in the one failure that was inspected, the model answered `90 [1]` and dropped the unit. Section 9 shows whether the new prompt fixed that, and sections 10 and 11 show what the model actually said when it still fails.

## 1. Get the latest code
This clones the repo the first time and pulls the newest version on later runs.

In [ ]:
REPO_URL = "https://github.com/azinanigiwu55-dotcom/docreader2.git"  # <- change if you forked it

import os, sys
REPO_DIR = REPO_URL.split("/")[-1].removesuffix(".git")
if os.path.basename(os.getcwd()) != REPO_DIR:
    if not os.path.exists(REPO_DIR):
        !git clone $REPO_URL
    %cd $REPO_DIR
!git pull -q
!pip install -q -e . sentence-transformers accelerate bitsandbytes

sys.path.insert(0, os.path.abspath("src"))
import docqa
print("docqa found at:", docqa.__file__)

## 2. (Optional) Save outputs to Google Drive
Colab wipes files when the session ends. Mounting Drive lets you keep results. Skip this cell if you do not need them.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
OUT = "/content/drive/MyDrive/private-doc-qa-results"
os.makedirs(OUT, exist_ok=True)

## 3. Choose the customer profile
A profile is a small JSON file that holds everything that differs between customers: name, tone, language, extra rules, refusal wording, vocabulary and retrieval settings. Two examples ship in `profiles/`.

Change `PROFILE_PATH` to try another one.

In [ ]:
from docqa.profile import load_profile
from docqa.prompts import build_system_prompt

PROFILE_PATH = "profiles/example_accounting_firm.json"   # or profiles/example_hr_helpdesk.json
profile = load_profile(PROFILE_PATH)

print("Customer :", profile.organisation or "(none)", f"[tenant: {profile.tenant_id}]")
print("Assistant:", profile.name, "|", profile.answer_style, "|", profile.language)
print("k =", profile.k, "| chunk =", profile.max_chars, "chars | default access:", profile.default_acl)
print("\n----- system prompt the model will receive -----")
print(build_system_prompt(profile))

## 4. Generate synthetic documents and questions
Fictional firm, fictional clients. Every question has a known answer, so evaluation needs no manual labelling.
`acl.json` says which user groups may read each document.

In [ ]:
from docqa.synth import write_all
print(write_all("data"))
!ls data/docs | head -12
!cat data/acl.json | head -12

## 5. Ingest, chunk and build the hybrid index
Chunk size and the default access rule now come from the profile.

In [ ]:
from docqa.ingest import ingest_folder, load_acl
from docqa.index import HybridIndex, SentenceTransformerEmbedder

chunks = ingest_folder("data/docs", load_acl("data/acl.json"),
                       default_acl=profile.default_acl,
                       max_chars=profile.max_chars, overlap=profile.overlap)
print(len(chunks), "chunks from", len({c.doc for c in chunks}), "documents")

embedder = SentenceTransformerEmbedder("BAAI/bge-small-en-v1.5")
index = HybridIndex(embedder).build(chunks)
index.save("data/index")

## 6. Search, and check permissions
The same query returns different results depending on who is asking. Restricted chunks are filtered out **before** ranking, so they can never reach the model.

The second cell shows the profile's **glossary** at work: the customer's own words widen the search without changing the question the model sees.

In [ ]:
q = "What is the annual audit fee for Karoo Fresh Produce (Pty) Ltd?"
for groups in (["partners"], ["all"]):
    print("\nUser groups:", groups)
    for h in index.search(q, k=3, user_groups=groups):
        print(f"  {h.score:.4f}  {h.chunk.doc} ({h.chunk.loc})")

In [ ]:
q = "What is the notice period for Karoo Fresh Produce?"
print("Question      :", q)
print("Search query  :", profile.expand_query(q))
print()
for h in index.search(profile.expand_query(q), k=3, user_groups=["partners"]):
    print(f"  {h.score:.4f}  {h.chunk.doc} ({h.chunk.loc})")

## 7. Load a language model
Start with the 3B model (fast, fits the free T4). The 7B model in 4-bit gives better answers, is slower, and still fits.

In [ ]:
from docqa.answer import HFBackend

MODEL = "3b"    # "3b" (fast) or "7b" (better, 4-bit, slower)

if MODEL == "3b":
    backend = HFBackend("Qwen/Qwen2.5-3B-Instruct")
    MODEL_LABEL = "Qwen2.5-3B"
else:
    backend = HFBackend("Qwen/Qwen2.5-7B-Instruct", load_in_4bit=True)
    MODEL_LABEL = "Qwen2.5-7B (4-bit)"
# For answer_style "detailed", allow longer replies: HFBackend(..., max_new_tokens=400)
print("Loaded", MODEL_LABEL)

## 8. Ask questions with citations
Each citation now carries the **supporting quote** from the source, which is what a chat UI would show as a source preview. If a figure in the answer is not in the cited sources, the answer is rejected with `reason=ungrounded`.

In [ ]:
from docqa.answer import answer

def ask(question, groups, backend=backend):
    a = answer(question, index, backend, user_groups=groups,
               user="notebook", audit_path="audit.jsonl", profile=profile)
    print(f"Q: {question}\nGroups: {groups}\nA: {a.text}")
    for c in a.citations:
        print(f"   [{c['n']}] {c['doc']} ({c['loc']})")
        if c.get("quote"):
            print(f"       \"{c['quote']}\"")
    if a.ungrounded:
        print("   rejected figures:", a.ungrounded)
    print(f"   found={a.found} reason={a.reason} ({a.latency_s:.1f}s)\n")
    return a

ask("What is the annual audit fee for Karoo Fresh Produce (Pty) Ltd in its engagement letter?", ["partners"])
ask("What is the annual audit fee for Karoo Fresh Produce (Pty) Ltd in its engagement letter?", ["all"])   # no access
ask("How many days written notice are required to terminate the services agreement with Karoo Fresh Produce (Pty) Ltd?", ["partners"])
ask("What is the minimum password length required by the firm?", ["all"])
_ = ask("What is the home address of Aisha Patel?", ["partners"])  # not in the documents

### The grounding check, with a model that lies
This stand-in "model" cites a real source but states a figure that is not in it. It needs no GPU. A real model should never be this wrong, but the check exists for the day one is.

In [ ]:
class LyingModel:
    def generate(self, question, hits, messages):
        return "The annual audit fee is R1 [1]."

a = ask("What is the annual audit fee for Karoo Fresh Produce (Pty) Ltd in its engagement letter?",
        ["partners"], backend=LyingModel())
assert a.reason == "ungrounded"

## 9. Run the evaluation
Reports pass rates per question kind: answerable, prompt-injection, unanswerable and forbidden (access control). Use `limit=10` for a quick first run.

**Compare with the first run** (original prompt, Qwen2.5-3B): answerable 31/34, injection 1/1, unanswerable 5/5, forbidden 3/3, citation correct 94%, 0 access leaks.

In [ ]:
import json
from docqa.eval import run_eval, load_questions, format_report, save_results

questions = load_questions("data/questions.json")
result = run_eval(index, backend, questions, limit=None, profile=profile)
print(format_report(result))

## 10. Inspect failures and save results
Look at the failures first. `reason` tells you what happened: `ungrounded` means a figure was not in the cited source, `no_citation` means the model forgot to cite, `not_found` means it said NOT_FOUND. `raw` is exactly what the model replied (the `answer` column is replaced by the refusal text when a reply is rejected), and `retrieved` is which documents it was shown.

In [ ]:
import pandas as pd
df = pd.DataFrame(result["rows"])
failures = df[~df["passed"]][["id", "kind", "question", "reason", "raw", "retrieved", "cited"]]
print(len(failures), "failure(s)")
pd.set_option("display.max_colwidth", 140)
display(failures)

tag = f"{profile.tenant_id}_{MODEL}"
try:
    save_results(result, f"{OUT}/results_{tag}.json")
    open(f"{OUT}/results_{tag}.md", "w").write(format_report(result))
    print("Saved to", OUT)
except NameError:
    print("Drive not mounted (section 2 skipped): results not saved.")

## 11. Why did it fail? Look at what the model saw
For each failure this prints the expected answer, what the model replied, and the text of the chunks it was given. Check two things: **is the answer in the chunks** (if not, it is a retrieval problem) and **did the model refuse or misread even though it is there** (then it is a prompt or model problem).

In [ ]:
for _, row in failures.iterrows():
    q = next(x for x in questions if x["id"] == row["id"])
    groups = q.get("user_groups") or ["partners"]
    hits = index.search(profile.expand_query(q["question"]), k=profile.k, user_groups=groups)
    print("=" * 90)
    print(row["id"], "|", q["question"])
    print("expected :", q["expected"], "in", q["doc"])
    print("model said:", repr(row["raw"]), "->", row["reason"])
    for i, h in enumerate(hits, 1):
        marker = "  <-- has the answer" if any(e.lower() in h.chunk.text.lower() for e in q["expected"]) else ""
        print(f"  [{i}] {h.chunk.doc} ({h.chunk.loc}){marker}")
print("=" * 90)

### Change one thing at a time
Re-runs the evaluation with one profile setting changed, so you can see which setting moves the score. Each run takes about a minute on the T4. Add your own variants to the dictionary.

In [ ]:
from dataclasses import replace

variants = {
    "profile as is":   profile,
    "no extra rules":  replace(profile, extra_instructions=[]),
    "k = 3":           replace(profile, k=3),
    "k = 6":           replace(profile, k=6),
}

rows = []
for name, p in variants.items():
    r = run_eval(index, backend, questions, profile=p, verbose=False)
    by = r["summary"]["by_kind"]
    failed = [x["id"] for x in r["rows"] if not x["passed"]]
    rows.append({"variant": name, "answerable": f"{by['answerable']['passed']}/{by['answerable']['n']}",
                 "overall": f"{r['summary']['overall_pass_rate']:.0%}", "failed ids": ", ".join(failed)})
display(pd.DataFrame(rows))

### Copy this row into the README results table

In [ ]:
by_kind = result["summary"]["by_kind"]
def passed(kind):
    e = by_kind.get(kind)
    return f"{e['passed']}/{e['n']}" if e else "-"

cit = by_kind["answerable"]["citation_correct_rate"]
print(f"| {MODEL_LABEL}, bge-small (profile prompt + grounding check) | {passed('answerable')} | "
      f"{passed('injection')} | {passed('unanswerable')} | {passed('forbidden')} | {cit:.0%} | "
      f"{result['summary']['latency_mean_s']}s |")

## 12. Make a profile for a new customer
Everything below runs without a GPU. The point: a new customer is a new JSON file, not new code.

In [ ]:
import json
from docqa.profile import Profile

my_profile = {
    "tenant_id": "acme-legal",
    "name": "Acme Legal Assistant",
    "organisation": "Acme Legal LLP",
    "domain": "matter files, contracts and the firm's precedent library",
    "audience": "fee earners",
    "answer_style": "standard",
    "tone": "formal and careful",
    "extra_instructions": ["Quote clause numbers whenever the source gives them."],
    "not_found_text": "Nothing in your accessible matter files answers this. Please ask the supervising partner.",
    "glossary": {"governing law": ["jurisdiction", "laws of"]},
    "default_acl": ["admin"],
}
json.dump(my_profile, open("my_profile.json", "w"), indent=2)

p = load_profile("my_profile.json")
print(build_system_prompt(p))

In [ ]:
# Typos in setting names fail loudly instead of being ignored:
try:
    Profile.from_dict({"anser_style": "concise"})
except ValueError as e:
    print("Rejected:", e)

## Next steps
1. Change one thing at a time (chunk size, `k`, model, profile rule) and re-run section 9. Record the numbers.
2. Paste the README row from section 10 into the results table.
3. For a real customer, write evaluation questions from their own (non-confidential) examples and run them with their profile.
4. Move the index to Postgres + pgvector once the pipeline is stable (see the README roadmap).